# ImageXpress Pico

The Molecular Devices ImageXpress Pico is an automated microscope that communicates over gRPC/SiLA 2. It supports brightfield and fluorescence imaging with multiple objectives and filter cubes.

| Model | PLR Name | Capabilities |
|---|---|---|
| ImageXpress Pico | `Pico` | Microscopy (brightfield, DAPI, GFP, RFP, Texas Red, Cy5) |

**Requirements:** `grpcio`, `numpy`, and optionally `Pillow` for TIFF decoding. Install with `pip install grpcio numpy Pillow`.

## Setup

Create the `Pico` device with the instrument's network address. Use the `objectives` and `filter_cubes` arguments to declare which optics are installed at each turret/wheel position -- the driver will configure the instrument on setup.

In [ ]:
from pylabrobot.molecular_devices.imageXpress.pico import Pico
from pylabrobot.capabilities.microscopy import ImagingMode, Objective

pico = Pico(
    name="pico",
    host="192.168.1.100",  # replace with your instrument's IP
    objectives={
        0: Objective.O_4X_PL_FL,
        1: Objective.O_10X_PL_FL,
        2: Objective.O_20X_PL_FL,
    },
    filter_cubes={
        0: ImagingMode.BRIGHTFIELD,
        1: ImagingMode.DAPI,
        2: ImagingMode.GFP,
    },
)
await pico.setup()

## Plate setup

The Pico derives labware geometry (well spacing, bottom thickness, etc.) from the PLR plate definition, so any plate resource works:

In [ ]:
from pylabrobot.resources import Cor_96_wellplate_360ul_Fb

plate = Cor_96_wellplate_360ul_Fb(name="plate")

## Imaging

The Pico exposes a {class}`~pylabrobot.capabilities.microscopy.microscopy.Microscopy` capability on `pico.microscope`. For the full API including auto-exposure and auto-focus, see [Microscopy](../../capabilities/microscopy).

### Brightfield capture

In [ ]:
result = await pico.microscope.capture(
    well=plate.get_well("A1"),
    mode=ImagingMode.BRIGHTFIELD,
    objective=Objective.O_10X_PL_FL,
    plate=plate,
    exposure_time=50.0,   # ms
    focal_height=5.0,     # mm
    gain=1.0,
)

print(f"Captured {len(result.images)} image(s)")
print(f"Exposure: {result.exposure_time} ms, focal height: {result.focal_height} mm")

### Fluorescence capture

In [ ]:
result = await pico.microscope.capture(
    well=plate.get_well("B3"),
    mode=ImagingMode.DAPI,
    objective=Objective.O_20X_PL_FL,
    plate=plate,
    exposure_time=100.0,
    focal_height=5.0,
    gain=1.0,
)

### Supported objectives and imaging modes

| Objective | PLR Enum |
|---|---|
| N PLAN 2.5x/0.07 | `Objective.O_2_5X_N_PLAN` |
| PL FLUOTAR 4x/0.13 | `Objective.O_4X_PL_FL` |
| PL FLUOTAR 10x/0.30 | `Objective.O_10X_PL_FL` |
| PL FLUOTAR 20x/0.40 | `Objective.O_20X_PL_FL` |
| PL FLUOTAR 40x/0.60 | `Objective.O_40X_PL_FL` |

| Imaging Mode | PLR Enum |
|---|---|
| Brightfield | `ImagingMode.BRIGHTFIELD` |
| DAPI | `ImagingMode.DAPI` |
| GFP | `ImagingMode.GFP` |
| RFP | `ImagingMode.RFP` |
| Texas Red | `ImagingMode.TEXAS_RED` |
| Cy5 | `ImagingMode.CY5` |

## Door control

Open and close the plate drawer via the driver:

In [ ]:
await pico.driver.open_door()
# load plate
await pico.driver.close_door()

## Objective maintenance

To physically swap an objective, enter maintenance mode for the turret position:

In [ ]:
await pico.microscope.backend.enter_objective_maintenance(position=0)
# swap the objective
await pico.microscope.backend.exit_objective_maintenance()

## Teardown

In [ ]:
await pico.stop()